# 📐 Notebook 3: Metric Computation — BLEU, CIDEr, METEOR, ROUGE-L

**Reads:** `captions_all_models.csv`  
**Writes:** `metrics_all_models.csv`, `metric_summary.csv` and metric figures

---

### Research purpose

This notebook computes **per-image lexical/reference-based caption metrics** so they can later be compared against the human semantic labels from NB04.

The experiment does **not** assume that a high metric score means semantic correctness. The actual semantic-validity claim is tested only after joining these scores with human labels in NB05+.

### Major-ready corrections

- BLEU is explicitly documented as **sentence-level BLEU-1…4 with smoothing**.
- CIDEr uses a self-contained implementation of the **core published TF-IDF n-gram cosine formulation**, with image-level document frequency and the standard ×10 scale.
- METEOR and ROUGE-L use the **best matching reference per image**.
- Input/schema validation fails loudly instead of silently converting malformed data to scores.
- Metric failures are represented as `NaN` and reported, rather than silently becoming genuine zero scores.
- Arbitrary "confidence" terminology is removed. BLEU-4 score bins are labelled as **exploratory score bins**.
- Final outputs retain one row per image and all source columns, enabling a clean join with human semantic annotations.

In [ ]:
# CELL 1 — Install dependencies

import subprocess, sys

packages = [
    "nltk",
    "rouge-score",
    "pandas",
    "numpy",
    "matplotlib",
    "tqdm",
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

import nltk
for resource in ["wordnet", "omw-1.4", "punkt"]:
    nltk.download(resource, quiet=True)

print("✅ Dependencies and NLTK resources ready.")

In [ ]:
# CELL 2 — Imports & config

import sys
import warnings
import random
import ast
import math
from collections import Counter
from pathlib import Path
from typing import List

sys.path.insert(0, str(Path(".")))
import config as cfg

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer

SEED = cfg.SEED
MODELS = list(cfg.MODELS_TO_RUN)
OUTPUT_DIR = cfg.OUTPUT_DIR
FIGURES_DIR = cfg.FIGURES_DIR

random.seed(SEED)
np.random.seed(SEED)

MODEL_CAP_COLS = {
    "blip": "blip_caption",
    "blip2": "blip2_caption",
    "ofa": "ofa_caption",
    "vit_gpt2": "vit_gpt2_caption",
}

# Exploratory threshold retained for downstream misalignment analysis.
# It is NOT a confidence estimate.
BLEU_HIGH_THRESHOLD = 0.20

print("✅ Config ready.")
print("Models:", MODELS)

In [ ]:
# CELL 3 — Load and validate caption DataFrame

cap_path = OUTPUT_DIR / "captions_all_models.csv"
if not cap_path.exists():
    raise FileNotFoundError(
        f"{cap_path} not found. Run 02_caption_generation.ipynb first."
    )

df = pd.read_csv(cap_path)

required_columns = {
    "image_id",
    "ground_truth_caps",
    "n_gt_captions",
    "primary_gt_caption",
}
missing = sorted(required_columns - set(df.columns))
if missing:
    raise ValueError(f"Missing required columns: {missing}")

if df["image_id"].isna().any():
    raise ValueError("image_id contains missing values.")

if df["image_id"].duplicated().any():
    dupes = df.loc[df["image_id"].duplicated(), "image_id"].tolist()[:10]
    raise ValueError(f"image_id must be unique. Example duplicates: {dupes}")

def parse_caps(val):
    if isinstance(val, list):
        caps = val
    else:
        try:
            parsed = ast.literal_eval(str(val))
        except (ValueError, SyntaxError) as exc:
            raise ValueError(f"Malformed ground_truth_caps value: {val!r}") from exc
        if not isinstance(parsed, list):
            raise ValueError(
                f"ground_truth_caps must decode to a list, got {type(parsed).__name__}"
            )
        caps = parsed

    caps = [str(c).strip() for c in caps if pd.notna(c) and str(c).strip()]
    if not caps:
        raise ValueError("Found an image with no valid ground-truth captions.")
    return caps

df["gt_caps_parsed"] = df["ground_truth_caps"].apply(parse_caps)
refs_list = df["gt_caps_parsed"].tolist()

if len(refs_list) != len(df):
    raise AssertionError("Reference list length does not match DataFrame length.")

missing_model_cols = {
    model: MODEL_CAP_COLS[model]
    for model in MODELS
    if MODEL_CAP_COLS[model] not in df.columns
}
if missing_model_cols:
    raise ValueError(
        "Missing caption columns for configured models: "
        + str(missing_model_cols)
    )

print(f"✅ Loaded {len(df):,} images.")
print(f"   Average GT captions/image: {df['n_gt_captions'].mean():.2f}")
print(df[["image_id", "scenario_category", "primary_gt_caption"]].head(3))

In [ ]:
print("Configured models:", MODELS)

## 📏 Metric Implementations

In [ ]:
# CELL 4 — BLEU (sentence-level, n=1..4)

sf = SmoothingFunction().method1

def compute_bleu_scores(hyps: List[str], refs_list: List[List[str]]):
    results = {f"bleu{n}": [] for n in range(1, 5)}

    for hyp, refs in zip(hyps, refs_list):
        hyp_tok = str(hyp).lower().split() if pd.notna(hyp) else []
        refs_tok = [
            str(r).lower().split()
            for r in refs
            if pd.notna(r) and str(r).strip()
        ]

        if not hyp_tok or not refs_tok:
            for key in results:
                results[key].append(np.nan)
            continue

        for n in range(1, 5):
            weights = tuple([1 / n] * n + [0] * (4 - n))
            try:
                score = sentence_bleu(
                    refs_tok,
                    hyp_tok,
                    weights=weights,
                    smoothing_function=sf,
                )
            except Exception as exc:
                warnings.warn(f"BLEU-{n} failed for one row: {exc}")
                score = np.nan

            results[f"bleu{n}"].append(score)

    return results

test = compute_bleu_scores(
    ["a man on a couch"],
    [["a person sitting on a sofa"]],
)
print("BLEU test:", {k: f"{v[0]:.3f}" for k, v in test.items()})

In [ ]:
# CELL 5 — CIDEr (self-contained core TF-IDF cosine formulation)

def _ngrams(tokens, n):
    return Counter(
        tuple(tokens[i : i + n])
        for i in range(len(tokens) - n + 1)
    )

def _tokenize(text):
    return str(text).lower().split() if pd.notna(text) else []

def _build_image_level_idf(refs_list, n):
    # Document frequency counts an n-gram once per image.
    df_count = Counter()

    for refs in refs_list:
        image_ngrams = set()
        for ref in refs:
            image_ngrams.update(_ngrams(_tokenize(ref), n).keys())
        for gram in image_ngrams:
            df_count[gram] += 1

    n_images = max(len(refs_list), 1)
    return {
        gram: math.log(n_images / freq)
        for gram, freq in df_count.items()
        if freq > 0
    }

def _tfidf_vector(counts, idf, vocabulary):
    return np.array(
        [counts.get(g, 0) * idf.get(g, 0.0) for g in vocabulary],
        dtype=float,
    )

def _cosine(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))

def compute_cider_scores(hyps, refs_list, max_n=4):
    # Per-image CIDEr-style score:
    # average cosine similarity to the image's references across n=1..4,
    # then multiply by 10 to use the standard CIDEr score scale.
    idfs = {
        n: _build_image_level_idf(refs_list, n)
        for n in range(1, max_n + 1)
    }

    scores = []

    for hyp, refs in zip(hyps, refs_list):
        htok = _tokenize(hyp)
        per_n = []

        for n in range(1, max_n + 1):
            hng = _ngrams(htok, n)
            ref_ngs = [_ngrams(_tokenize(r), n) for r in refs]

            vocabulary = set(hng)
            for rng in ref_ngs:
                vocabulary.update(rng)

            if not vocabulary:
                per_n.append(0.0)
                continue

            vocabulary = list(vocabulary)
            hv = _tfidf_vector(hng, idfs[n], vocabulary)

            similarities = []
            for rng in ref_ngs:
                rv = _tfidf_vector(rng, idfs[n], vocabulary)
                similarities.append(_cosine(hv, rv))

            per_n.append(float(np.mean(similarities)))

        scores.append(10.0 * float(np.mean(per_n)))

    return scores

test_cider = compute_cider_scores(
    ["a man on a couch"],
    [["a person sitting on a sofa"]],
)
print(f"CIDEr test: {test_cider[0]:.3f}")

In [ ]:
# CELL 6 — METEOR + ROUGE-L

def compute_meteor_scores(hyps, refs_list):
    scores = []

    for hyp, refs in zip(hyps, refs_list):
        ht = _tokenize(hyp)
        rts = [_tokenize(r) for r in refs if pd.notna(r)]

        if not ht or not rts:
            scores.append(np.nan)
            continue

        try:
            score = max(meteor_score([rt], ht) for rt in rts)
        except Exception as exc:
            warnings.warn(f"METEOR failed for one row: {exc}")
            score = np.nan

        scores.append(score)

    return scores

def compute_rougeL_scores(hyps, refs_list):
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    scores = []

    for hyp, refs in zip(hyps, refs_list):
        hs = str(hyp).lower() if pd.notna(hyp) else ""

        if not hs.strip() or not refs:
            scores.append(np.nan)
            continue

        try:
            best = max(
                scorer.score(str(r).lower(), hs)["rougeL"].fmeasure
                for r in refs
                if pd.notna(r)
            )
        except Exception as exc:
            warnings.warn(f"ROUGE-L failed for one row: {exc}")
            best = np.nan

        scores.append(best)

    return scores

print(
    "METEOR test:",
    compute_meteor_scores(
        ["a man on a couch"],
        [["a person sitting on a sofa"]],
    )[0],
)
print(
    "ROUGE-L test:",
    compute_rougeL_scores(
        ["a man on a couch"],
        [["a person sitting on a sofa"]],
    )[0],
)

## 🔄 Compute All Metrics for All Models

In [ ]:
# CELL 7 — Main metric computation loop

print("🔄 Computing metrics...")

for model in tqdm(MODELS, desc="Models"):
    col = MODEL_CAP_COLS[model]
    hyps = df[col].fillna("").tolist()

    bleu = compute_bleu_scores(hyps, refs_list)
    for n in range(1, 5):
        df[f"bleu{n}_{model}"] = bleu[f"bleu{n}"]

    df[f"cider_{model}"] = compute_cider_scores(hyps, refs_list)
    df[f"meteor_{model}"] = compute_meteor_scores(hyps, refs_list)
    df[f"rougeL_{model}"] = compute_rougeL_scores(hyps, refs_list)

    # Exploratory BLEU-4 score bin — NOT model confidence.
    df[f"bleu4_score_bin_{model}"] = pd.cut(
        df[f"bleu4_{model}"],
        bins=[-np.inf, 0.10, 0.30, np.inf],
        labels=["low", "medium", "high"],
        right=False,
    )

    metric_cols = [
        f"bleu4_{model}",
        f"cider_{model}",
        f"meteor_{model}",
        f"rougeL_{model}",
    ]
    invalid_count = int(df[metric_cols].isna().sum().sum())

    print(
        f"  {model.upper()}: "
        f"BLEU4={df[f'bleu4_{model}'].mean():.4f}  "
        f"CIDEr={df[f'cider_{model}'].mean():.4f}  "
        f"METEOR={df[f'meteor_{model}'].mean():.4f}  "
        f"ROUGE-L={df[f'rougeL_{model}'].mean():.4f}  "
        f"(NaN cells={invalid_count})"
    )

print("\n✅ All configured model metrics computed.")

In [ ]:
# CELL 8 — Summary table

rows = []

for model in MODELS:
    row = {
        "Model": cfg.MODEL_DISPLAY.get(model, model.upper())
    }

    for metric in ["bleu1", "bleu4", "cider", "meteor", "rougeL"]:
        col = f"{metric}_{model}"

        if col in df.columns:
            row[metric.upper()] = (
                f"{df[col].mean():.4f} ± {df[col].std():.4f}"
            )
        else:
            row[metric.upper()] = "N/A"

    rows.append(row)

summary = pd.DataFrame(rows).set_index("Model")

print("\n📊 Metric Summary (mean ± std):")
print(summary.to_string())

summary.to_csv(OUTPUT_DIR / "metric_summary.csv")
print("\n💾 metric_summary.csv saved.")

In [ ]:
# CELL 9 — BLEU-4 distribution plots

available_models = [
    m for m in MODELS
    if f"bleu4_{m}" in df.columns
]

fig, axes = plt.subplots(
    1,
    len(available_models),
    figsize=(4 * max(len(available_models), 1), 4),
    squeeze=False,
)
axes = axes.ravel()

fig.suptitle(
    "BLEU-4 Score Distribution per Model",
    fontsize=13,
    fontweight="bold",
)

for ax, model in zip(axes, available_models):
    sc = df[f"bleu4_{model}"].dropna()

    ax.hist(sc, bins=20, edgecolor="white", alpha=0.85)
    ax.axvline(
        sc.mean(),
        lw=2,
        label=f"mean={sc.mean():.3f}",
    )
    ax.axvline(
        BLEU_HIGH_THRESHOLD,
        lw=1.5,
        linestyle="--",
        label=f"exploratory threshold={BLEU_HIGH_THRESHOLD}",
    )

    ax.set_title(
        cfg.MODEL_DISPLAY.get(model, model.upper()),
        fontweight="bold",
    )
    ax.set_xlabel("Sentence-level BLEU-4")
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(
    FIGURES_DIR / "bleu4_distribution.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()
print("📊 Saved.")

In [ ]:
# CELL 10 — Cross-metric correlation heatmaps

available_models = [
    m for m in MODELS
    if all(
        f"{metric}_{m}" in df.columns
        for metric in ["bleu1", "bleu4", "cider", "meteor", "rougeL"]
    )
]

if not available_models:
    raise ValueError("No complete model metric sets available.")

fig, axes = plt.subplots(
    1,
    len(available_models),
    figsize=(5 * len(available_models), 5),
    squeeze=False,
)
axes = axes.ravel()

fig.suptitle(
    "Cross-Metric Correlation per Model",
    fontsize=13,
    fontweight="bold",
)

for ax, model in zip(axes, available_models):
    cols = [
        f"{m}_{model}"
        for m in ["bleu1", "bleu4", "cider", "meteor", "rougeL"]
    ]

    corr = df[cols].corr()
    corr.index = corr.columns = [
        c.replace(f"_{model}", "") for c in cols
    ]

    im = ax.imshow(corr, vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr.columns)))
    ax.set_yticks(range(len(corr.index)))
    ax.set_xticklabels(corr.columns, rotation=45, ha="right")
    ax.set_yticklabels(corr.index)
    ax.set_title(
        cfg.MODEL_DISPLAY.get(model, model.upper()),
        fontweight="bold",
    )

    for i in range(len(corr.index)):
        for j in range(len(corr.columns)):
            ax.text(
                j, i, f"{corr.iloc[i, j]:.2f}",
                ha="center", va="center",
            )

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
fig.savefig(
    FIGURES_DIR / "cross_metric_correlation.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()
print("📊 Saved.")

print(
    "\nNOTE: these are correlations BETWEEN automatic metrics. "
    "They do not test semantic correctness; that requires human labels."
)

In [ ]:
# CELL 11 — Box plots across models

metrics_plot = ["bleu4", "cider", "meteor", "rougeL"]

fig, axes = plt.subplots(
    1,
    len(metrics_plot),
    figsize=(4 * len(metrics_plot), 5),
)

fig.suptitle(
    "Metric Score Distributions Across Models",
    fontsize=13,
    fontweight="bold",
)

for ax, metric in zip(axes, metrics_plot):
    data = {
        cfg.MODEL_DISPLAY.get(m, m): df[f"{metric}_{m}"].dropna().values
        for m in MODELS
        if f"{metric}_{m}" in df.columns
    }

    if not data:
        continue

    ax.boxplot(
        list(data.values()),
        labels=list(data.keys()),
        patch_artist=True,
        notch=True,
    )

    ax.set_title(metric.upper(), fontweight="bold")
    ax.set_ylabel("Score")
    ax.tick_params(axis="x", rotation=15)
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
fig.savefig(
    FIGURES_DIR / "metric_boxplots.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()
print("📊 Saved.")

In [ ]:
# CELL 12 — Final validation and save

metric_prefixes = ["bleu", "cider", "meteor", "rougeL"]
metric_cols = [
    c for c in df.columns
    if any(c.startswith(prefix) for prefix in metric_prefixes)
]

if not metric_cols:
    raise RuntimeError("No metric columns were generated.")

if len(df) != df["image_id"].nunique():
    raise AssertionError("Output no longer has one row per image.")

nan_report = df[metric_cols].isna().sum()
nan_report = nan_report[nan_report > 0]

if len(nan_report):
    print("⚠️ Metric NaNs detected:")
    print(nan_report.to_string())
else:
    print("✅ No metric NaNs detected.")

out_path = OUTPUT_DIR / "metrics_all_models.csv"
df.to_csv(out_path, index=False)

print(f"\n💾 metrics_all_models.csv saved: {out_path}")
print(f"   Metric-related columns added: {len(metric_cols)}")
print(f"   Rows preserved: {len(df):,}")
print("\n✅ Notebook 3 COMPLETE — run 04_human_annotation.ipynb next.")